In [ ]:
# %%
# Импорты и настройки

import gc
import json
import math
from pathlib import Path

import numpy as np
import polars as pl

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score


HORIZON = 24.0

TO_DROP = [
    "ид_события",
    "ид_канала_данных",
    "дата",
    "время",
    "datetime",
]

TARGET_COLS = [
    "target_lower",
    "target_upper",
]


# Чанки можно задавать отдельно
# train прорежен в ноутбуке 2, поэтому берём каждый 3-й чанк, а не первые 4
N_CHUNKS = len(list(Path("data/final_train").glob("chunk_*.parquet")))
TRAIN_CHUNKS = [*range(0, N_CHUNKS, 4)]  # четверть прореженного train
VAL_CHUNKS = [*range(N_CHUNKS)]
TEST_CHUNKS = []  # слепые наборы в отборе не участвуют


DROP_PER_STEP = 20
MIN_FEATURES = 10

# Сколько строк validation использовать для SHAP
SHAP_ROWS = 50_000


OUTPUT_DIR = Path("feature_selection_shap_rfe")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HISTORY_PATH = OUTPUT_DIR / "history.csv"
SELECTED_STEP_PATH = OUTPUT_DIR / "selected_step.json"
SENSOR_COL = "тип_датчика"


with open("data/incident_by_sensor.json", encoding="utf-8") as f:
    INCIDENT_BY_SENSOR: dict[str, str] = json.load(f)


def macro_pr_auc(
    y: np.ndarray,
    p: np.ndarray,
    sensor: np.ndarray,
) -> float:
    """Среднее PR-AUC по 4 классам: GAS (88% строк) не перевешивает остальных."""
    incident = pl.Series(sensor).replace_strict(
        INCIDENT_BY_SENSOR, default=None
    ).to_numpy()
    scores = [
        average_precision_score(y[incident == c], p[incident == c])
        for c in sorted(set(INCIDENT_BY_SENSOR.values()))
        if y[incident == c].any()
    ]
    return float(np.mean(scores))


MODEL_PARAMS = {
    "loss_function": "Logloss",
    "eval_metric": "PRAUC:type=Classic",
    "iterations": 100,
    "depth": 6,
    "learning_rate": 0.05,
    "random_seed": 42,
    "verbose": 1,
}

In [ ]:
# %%
# LazyFrame для нужного split

def scan_split(
    split: str,
    chunks: list[int],
) -> pl.LazyFrame:

    files = [
        f"data/final_{split}/chunk_{i:05d}.parquet"
        for i in chunks
    ]

    return (
        pl.scan_parquet(files)
        .drop(TO_DROP)
        .with_columns(
            (
                (pl.col("target_upper") != -1)
                & (pl.col("target_lower") <= HORIZON)
            )
            .cast(pl.Int8)
            .alias("target")
        )
    )


# Схему берем без загрузки данных
train_schema = scan_split(
    "train",
    [TRAIN_CHUNKS[0]],
).collect_schema()


feature_cols = [
    c
    for c in train_schema.names()
    if c not in TARGET_COLS + ["target"]
]


cat_features = [
    c
    for c in feature_cols
    if train_schema[c] in (
        pl.String,
        pl.Categorical,
        pl.Enum,
    )
]


print(f"Features:  {len(feature_cols)}")
print(f"Cat feats: {len(cat_features)}")

In [ ]:
# %%
# Служебные функции

def collect_df(
    split: str,
    chunks: list[int],
    features: list[str],
    cat_features_current: list[str],
    with_target: bool = True,
    head: int | None = None,
) -> pl.DataFrame:

    cols = [*dict.fromkeys(
        features + (["target", SENSOR_COL] if with_target else [])
    )]

    lf = (
        scan_split(split, chunks)
        .select(cols)
    )

    # Null в категориях заполняем до collect
    if cat_features_current:
        lf = lf.with_columns(
            pl.col(cat_features_current).fill_null("null")
        )

    if head is not None:
        lf = lf.head(head)

    return lf.collect(
        engine="streaming"
    )


def make_pool(
    split: str,
    chunks: list[int],
    features: list[str],
    cat_features_current: list[str],
    return_y: bool = False,
) -> Pool | tuple[Pool, np.ndarray]:

    df = collect_df(
        split=split,
        chunks=chunks,
        features=features,
        cat_features_current=cat_features_current,
        with_target=True,
    )

    y = df["target"].to_numpy()
    sensor = df[SENSOR_COL].to_numpy()

    pool = Pool(
        data=df.select(features),
        label=y,
        cat_features=cat_features_current,
    )

    del df
    gc.collect()

    if return_y:
        return pool, y, sensor

    del y
    gc.collect()

    return pool

In [ ]:
# %%
# Test score считаем по чанкам
# В RAM одновременно лежит только один test chunk

def get_test_pr_auc(
    model: CatBoostClassifier,
    features: list[str],
    cat_features_current: list[str],
) -> float:

    if not TEST_CHUNKS:
        return float("nan")

    y_parts = []
    p_parts = []

    for chunk in TEST_CHUNKS:

        test_pool, y_chunk, _ = make_pool(
            split="test",
            chunks=[chunk],
            features=features,
            cat_features_current=cat_features_current,
            return_y=True,
        )

        p_chunk = model.predict_proba(
            test_pool
        )[:, 1]

        y_parts.append(y_chunk)
        p_parts.append(p_chunk)

        del test_pool
        del y_chunk
        del p_chunk

        gc.collect()


    # Для точного PR-AUC нужны prediction по всему test
    # Это только два одномерных массива, поэтому памяти мало
    y_test = np.concatenate(y_parts)
    p_test = np.concatenate(p_parts)

    score = average_precision_score(
        y_test,
        p_test,
    )

    del y_parts
    del p_parts
    del y_test
    del p_test

    gc.collect()

    return score

In [ ]:
# %%
# SHAP считаем частями по validation chunks
# Полная SHAP-матрица для всего validation в RAM не создается

def get_mean_abs_shap(
    model: CatBoostClassifier,
    features: list[str],
    cat_features_current: list[str],
) -> np.ndarray:

    shap_sum = np.zeros(
        len(features),
        dtype=np.float64,
    )

    total_rows = 0

    rows_per_chunk = math.ceil(
        SHAP_ROWS / len(VAL_CHUNKS)
    )


    for chunk in VAL_CHUNKS:

        shap_df = collect_df(
            split="val",
            chunks=[chunk],
            features=features,
            cat_features_current=cat_features_current,
            with_target=False,
            head=rows_per_chunk,
        )

        if shap_df.height == 0:
            del shap_df
            gc.collect()
            continue


        shap_pool = Pool(
            data=shap_df,
            cat_features=cat_features_current,
        )

        del shap_df
        gc.collect()


        shap_values = model.get_feature_importance(
            shap_pool,
            type="ShapValues",
        )


        shap_sum += np.abs(
            shap_values[:, :-1]
        ).sum(axis=0)

        total_rows += shap_values.shape[0]


        del shap_pool
        del shap_values

        gc.collect()


    if total_rows == 0:
        raise ValueError("Не удалось получить строки для SHAP")


    return shap_sum / total_rows

In [ ]:
# %%
# Feature selection
# На каждом шаге:
# 1. Загружаем только текущие фичи
# 2. Обучаем модель
# 3. Сразу удаляем train/val Pool
# 4. Считаем test по чанкам
# 5. Считаем SHAP на ограниченной части validation
# 6. Убираем 10 худших фичей

history = []

current_features = feature_cols.copy()

removed_this_step = []

step = 0


while True:

    print()
    print("=" * 80)
    print(
        f"step={step} | "
        f"features={len(current_features)}"
    )


    current_cat_features = [
        c
        for c in cat_features
        if c in current_features
    ]


    # Train Pool
    train_pool = make_pool(
        split="train",
        chunks=TRAIN_CHUNKS,
        features=current_features,
        cat_features_current=current_cat_features,
        return_y=False,
    )


    # Validation Pool
    val_pool, y_val, sensor_val = make_pool(
        split="val",
        chunks=VAL_CHUNKS,
        features=current_features,
        cat_features_current=current_cat_features,
        return_y=True,
    )


    # Обучение
    model = CatBoostClassifier(
        **MODEL_PARAMS
    )

    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=50,
    )


    # Validation score
    p_val = model.predict_proba(
        val_pool
    )[:, 1]

    val_pr_auc = average_precision_score(
        y_val,
        p_val,
    )

    val_macro_pr_auc = macro_pr_auc(
        y_val,
        p_val,
        sensor_val,
    )


    # Train и validation больше не нужны
    del train_pool
    del val_pool
    del y_val
    del p_val
    del sensor_val

    gc.collect()


    # Test score по одному чанку
    test_pr_auc = get_test_pr_auc(
        model=model,
        features=current_features,
        cat_features_current=current_cat_features,
    )


    print(
        f"val={val_pr_auc:.6f} | "
        f"val_macro={val_macro_pr_auc:.6f} | "
        f"test={test_pr_auc:.6f}"
    )


    # Сохраняем результат шага
    history.append({
        "step": step,
        "n_features": len(current_features),
        "val_pr_auc": val_pr_auc,
        "val_macro_pr_auc": val_macro_pr_auc,
        "test_pr_auc": test_pr_auc,
        "best_iteration": model.get_best_iteration(),
        "removed_features": " | ".join(
            removed_this_step
        ),
    })


    pl.DataFrame(
        history
    ).write_csv(
        HISTORY_PATH
    )


    # Если дошли до минимального числа фичей
    if len(current_features) <= MIN_FEATURES:

        del model

        gc.collect()

        break


    # SHAP на ограниченной части validation
    mean_abs_shap = get_mean_abs_shap(
        model=model,
        features=current_features,
        cat_features_current=current_cat_features,
    )


    shap_importance = (
        pl.DataFrame({
            "feature": current_features,
            "mean_abs_shap": mean_abs_shap,
        })
        .sort("mean_abs_shap")
    )


    n_drop = min(
        DROP_PER_STEP,
        len(current_features) - MIN_FEATURES,
    )


    removed_this_step = (
        shap_importance
        .head(n_drop)
        ["feature"]
        .to_list()
    )


    print("remove:")

    for feature in removed_this_step:
        print(f"# {feature}")


    removed_set = set(
        removed_this_step
    )


    current_features = [
        c
        for c in current_features
        if c not in removed_set
    ]


    step += 1


    del mean_abs_shap
    del shap_importance
    del model

    gc.collect()

In [ ]:
# %%
# Итоговая таблица

history_df = pl.read_csv(
    HISTORY_PATH
)

history_df

In [ ]:
history_df.sort("val_macro_pr_auc", descending=True)

In [ ]:
# %%
# Берем набор фичей на выбранном шаге

history_df = pl.read_csv(HISTORY_PATH)

# Минимум фичей при macro PR-AUC на val не хуже лучшего шага на 0.005
best_macro = history_df["val_macro_pr_auc"].max()
SELECTED_STEP = (
    history_df
    .filter(pl.col("val_macro_pr_auc") >= best_macro - 0.005)
    .sort("n_features")
    ["step"][0]
)
SELECTED_STEP_PATH.write_text(json.dumps({"step": int(SELECTED_STEP)}))

removed_features = (
    history_df
    .filter(pl.col("step") <= SELECTED_STEP)
    .select("removed_features")
    .drop_nulls()
    ["removed_features"]
    .to_list()
)

removed_features = {
    feature
    for row in removed_features
    for feature in row.split(" | ")
    if feature
}

selected_features = [
    feature
    for feature in feature_cols
    if feature not in removed_features
]

print(f"Step: {SELECTED_STEP}")
print(f"Features: {len(selected_features)}")

selected_features

In [ ]:
# %%
def get_n_rows(
    split: str,
    chunks: list[int],
) -> int:

    files = [
        f"data/final_{split}/chunk_{i:05d}.parquet"
        for i in chunks
    ]

    return (
        pl.scan_parquet(files)
        .select(pl.len())
        .collect()
        .item()
    )


n_train = get_n_rows(
    "train",
    TRAIN_CHUNKS,
)

n_val = get_n_rows(
    "val",
    VAL_CHUNKS,
)

n_test = 0  # test не используется


print(f"Train: {n_train:,}")
print(f"Val:   {n_val:,}")
print(f"Test:  {n_test:,}")